In [1]:
import sys
import importlib.metadata as metadata

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)

for package in ["torch", "transformers", "safetensors", "sentencepiece"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: Not installed")

Python version: 3.11.9
Python executable: /Users/catherinexie/Documents/kgl-mockproject/.venv/bin/python
torch: Not installed
transformers: Not installed
safetensors: Not installed
sentencepiece: Not installed


In [1]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Kernel is working!")

PyTorch: 2.1.1
Transformers: 4.35.2
Kernel is working!


In [2]:
import torch
import transformers
import safetensors
import numpy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Safetensors:", safetensors.__version__)
print("NumPy:", numpy.__version__)
print("Apple Silicon GPU available:", torch.backends.mps.is_available())

PyTorch: 2.1.1
Transformers: 4.35.2
Safetensors: 0.4.0
NumPy: 1.26.4
Apple Silicon GPU available: True


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "tum-nlp/Deberta_Human_Value_Detector"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pretrained classifier
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True
)

# Set evaluation mode
model.eval()

print("Model loaded successfully!")

/Users/catherinexie/Documents/kgl-mockproject/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/tum-nlp/Deberta_Human_Value_Detector:
- modeling_deberta_arg_classifier.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Model loaded successfully!


In [4]:
example_text = (
    "We must protect our country's independence "
    "and ensure the security of our citizens."
)

inputs = tokenizer(
    example_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    prediction = model(
        inputs["input_ids"],
        inputs["attention_mask"]
    )["output"].flatten()

print("Number of value scores:", len(prediction))
print("First 5 scores:", prediction[:5].tolist())

Number of value scores: 20
First 5 scores: [0.114350326359272, 0.7895858883857727, 0.026197031140327454, 0.005079697351902723, 0.015388639643788338]


In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

contexts = pd.read_csv(
    PROJECT_ROOT / "data/contexts/focal_self_assertion_contexts.csv"
)

VALUE_LABELS = [
    "Self-direction: thought",
    "Self-direction: action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power: dominance",
    "Power: resources",
    "Face",
    "Security: personal",
    "Security: societal",
    "Tradition",
    "Conformity: rules",
    "Conformity: interpersonal",
    "Humility",
    "Benevolence: caring",
    "Benevolence: dependability",
    "Universalism: concern",
    "Universalism: nature",
    "Universalism: tolerance",
    "Universalism: objectivity"
]

assert len(contexts) == 1230
assert len(VALUE_LABELS) == 20

print("Passages ready:", len(contexts))
print("Value categories:", len(VALUE_LABELS))

Passages ready: 1230
Value categories: 20


In [6]:
import numpy as np
import torch

BATCH_SIZE = 8

all_predictions = []

model.eval()

for start in range(0, len(contexts), BATCH_SIZE):
    batch_texts = contexts["sentence_text"].iloc[
        start:start + BATCH_SIZE
    ].tolist()

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.inference_mode():
        predictions = model(
            inputs["input_ids"],
            inputs["attention_mask"]
        )["output"]

    all_predictions.append(predictions.cpu().numpy())

    if (start // BATCH_SIZE + 1) % 20 == 0:
        print(f"Processed {min(start + BATCH_SIZE, len(contexts))}/{len(contexts)}")

value_matrix = np.concatenate(all_predictions, axis=0)

assert value_matrix.shape == (1230, 20)
assert np.isfinite(value_matrix).all()

print("Classification complete:", value_matrix.shape)

Processed 160/1230
Processed 320/1230
Processed 480/1230
Processed 640/1230
Processed 800/1230
Processed 960/1230
Processed 1120/1230
Classification complete: (1230, 20)


In [7]:
value_scores = pd.DataFrame(
    value_matrix,
    columns=VALUE_LABELS
)

classified = pd.concat(
    [contexts.reset_index(drop=True), value_scores],
    axis=1
)

output_path = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "focal_self_assertion_values.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

classified.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Output shape:", classified.shape)

display(classified.head(3))

Saved: /Users/catherinexie/Documents/kgl-mockproject/data/results/focal_self_assertion_values.csv
Output shape: (1230, 27)


,iso3,year,filename,sentence_id,sentence_text,matched_terms,n_matches,Self-direction: thought,Self-direction: action,Stimulation,...,Tradition,Conformity: rules,Conformity: interpersonal,Humility,Benevolence: caring,Benevolence: dependability,Universalism: concern,Universalism: nature,Universalism: tolerance,Universalism: objectivity
0,CHL,2016,CHL_71_2016.txt,22,I insist that we must act now.,"[""insist""]",1,0.042132,0.091307,0.036236,...,0.013745,0.444692,0.003577,0.058474,0.227511,0.233442,0.013169,0.024272,0.051672,0.209250
1,CHL,2016,CHL_71_2016.txt,28,I am also speaking from experience of having e...,"[""requested""]",1,0.040901,0.545871,0.035547,...,0.007977,0.021869,0.005527,0.033556,0.215697,0.211969,0.087616,0.003457,0.208381,0.187872
2,CHL,2016,CHL_71_2016.txt,38,Our nation’s principles are immutable and incl...,"[""call""]",1,0.219084,0.406526,0.008782,...,0.713527,0.892252,0.014523,0.010779,0.015576,0.116845,0.827891,0.006263,0.089471,0.053761


In [8]:
import numpy as np

# Reproducible sample: 60 sentences, balanced across countries
validation = (
    classified.groupby("iso3", group_keys=False)
    .sample(n=3, random_state=42)
)

# Add 6 more sentences from those not selected
remaining = classified.drop(validation.index)
extra = remaining.sample(n=6, random_state=42)

validation = pd.concat([validation, extra]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

# Identify the model's three highest-scoring values per sentence
value_cols = VALUE_LABELS

validation["top_3_values"] = validation[value_cols].apply(
    lambda row: [
        (label, round(float(row[label]), 3))
        for label in row.nlargest(3).index
    ],
    axis=1
)

display(
    validation[
        ["iso3", "year", "sentence_text", "top_3_values"]
    ]
)

,iso3,year,sentence_text,top_3_values
0,CHL,2022,"In that electoral event, we citizens clearly r...","[(Power: dominance, 0.698), (Self-direction: a..."
1,CHN,2023,China supports all efforts that are conducive ...,"[(Universalism: concern, 0.963), (Security: so..."
2,POL,2020,In the name of SOLIDARITY we should all speak ...,"[(Universalism: concern, 0.962), (Conformity: ..."
3,THA,2023,Thailand is ready to form regional and global ...,"[(Universalism: nature, 0.971), (Achievement, ..."
4,GHA,2024,"As my presidency draws to a close, I want to a...","[(Benevolence: dependability, 0.726), (Univers..."
5,CHN,2018,We need to incorporate its implementation into...,"[(Universalism: nature, 0.961), (Achievement, ..."
6,PAK,2025,"Pakistan stands ready for a composite, compreh...","[(Achievement, 0.798), (Power: dominance, 0.75..."
7,TJK,2024,"Taking this opportunity, we call on partner co...","[(Power: dominance, 0.771), (Achievement, 0.74..."
8,GHA,2023,"We cannot continue to preach democracy, equali...","[(Universalism: concern, 0.998), (Security: so..."
9,GBR,2024,And our willingness to defend them.,"[(Security: societal, 0.993), (Universalism: c..."


In [9]:
output_path = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "value_validation_sample.csv"
)

validation.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/catherinexie/Documents/kgl-mockproject/data/results/value_validation_sample.csv
